# 画像認識と物体検出

画像認識には、画像全体にラベルを付ける分類、物体の位置を箱で返す検出、画素単位で領域を返すセグメンテーションがあります。YOLO は物体検出の代表的な設計です。画像を一度の forward pass でグリッドごとの予測に変換し、座標、物体らしさ、クラス確率を同時に出します。

## 検出は出力形式から複雑になる

分類の出力はクラス確率だけです。検出では、どのクラスかに加えて、どこにあるか、どれくらい確信しているか、重複した箱をどう整理するかまで出力に含まれます。箱、IoU、NMS、AP を実行で確かめると、物体検出の出力がどの手順で評価に変わるかが分かります。

In [ ]:
import math
import random

random.seed(23)

def round_list(values, digits=3):
    return [round(float(v), digits) for v in values]

IMAGE_SIZE = 96
CLASSES = ['circle', 'square', 'triangle']

## 箱の表現をそろえる

物体検出では、同じ箱でも複数の表し方を使います。xyxy は左上と右下の座標です。xywh は中心座標と幅・高さです。損失や後処理の式を読むためには、この変換を確実に扱える必要があります。

In [ ]:
def xywh_to_xyxy(box):
    cx, cy, w, h = box
    return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


def xyxy_to_xywh(box):
    x1, y1, x2, y2 = box
    return [(x1 + x2) / 2, (y1 + y2) / 2, max(0.0, x2 - x1), max(0.0, y2 - y1)]


def clip_box(box, image_size=IMAGE_SIZE):
    x1, y1, x2, y2 = box
    return [
        min(max(x1, 0.0), image_size),
        min(max(y1, 0.0), image_size),
        min(max(x2, 0.0), image_size),
        min(max(y2, 0.0), image_size),
    ]

sample_xywh = [48.0, 36.0, 30.0, 18.0]
converted = xywh_to_xyxy(sample_xywh)
restored = xyxy_to_xywh(converted)
print('xywh:', round_list(sample_xywh))
print('xyxy:', round_list(converted))
print('restored:', round_list(restored))

## IoU は位置の当たり具合を測る

Intersection over Union は、予測箱と正解箱の重なり面積を和集合面積で割った値です。クラスが合っていても、IoU が低ければ位置を外しています。検出の正解判定では、IoU が 0.5 以上などのしきい値を使います。

In [ ]:
def box_area(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter = [max(ax1, bx1), max(ay1, by1), min(ax2, bx2), min(ay2, by2)]
    inter_area = box_area(inter)
    union = box_area(a) + box_area(b) - inter_area
    return 0.0 if union <= 0 else inter_area / union

truth = [20.0, 22.0, 60.0, 58.0]
pred_good = [22.0, 24.0, 62.0, 56.0]
pred_shifted = [44.0, 23.0, 82.0, 55.0]
pred_miss = [65.0, 66.0, 90.0, 90.0]
for name, box in [('good', pred_good), ('shifted', pred_shifted), ('miss', pred_miss)]:
    print(name, 'IoU=', round(iou_xyxy(box, truth), 4))

## NMS は同じ物体への重複予測を減らす

検出器は同じ物体の近くに複数の箱を出します。Non-Maximum Suppression は、スコアが高い箱から残し、強く重なる低スコア箱を捨てます。クラスが違う物体まで消さないよう、通常はクラスごとに行います。

In [ ]:
def nms_single_class(detections, iou_threshold=0.5):
    ordered = sorted(detections, key=lambda d: d['score'], reverse=True)
    kept = []
    while ordered:
        best = ordered.pop(0)
        kept.append(best)
        ordered = [d for d in ordered if iou_xyxy(best['box'], d['box']) < iou_threshold]
    return kept

raw_detections = [
    {'box': [18, 20, 60, 58], 'score': 0.92, 'class_id': 1},
    {'box': [20, 22, 61, 57], 'score': 0.88, 'class_id': 1},
    {'box': [44, 22, 82, 56], 'score': 0.64, 'class_id': 1},
    {'box': [10, 64, 34, 88], 'score': 0.73, 'class_id': 0},
]
kept_square = nms_single_class([d for d in raw_detections if d['class_id'] == 1], iou_threshold=0.45)
print('kept square boxes:')
for det in kept_square:
    print(det['score'], round_list(det['box'], 1))

## YOLO はグリッドごとに責任を持つ

YOLO 系の検出器は画像を S x S のセルへ分けます。物体の中心が入ったセルが、その物体を担当します。各セルは、複数のアンカーごとに座標と objectness を出し、セル単位またはアンカー単位でクラスを出します。

In [ ]:
def wh_iou(box_wh, anchor_wh):
    bw, bh = box_wh
    aw, ah = anchor_wh
    inter = min(bw, aw) * min(bh, ah)
    union = bw * bh + aw * ah - inter
    return 0.0 if union <= 0 else inter / union


def encode_yolo_target(gt_box, class_id, image_size=IMAGE_SIZE, grid=6, anchors=None, num_classes=3):
    if anchors is None:
        anchors = [[16.0, 16.0], [32.0, 24.0], [48.0, 40.0]]
    num_anchors = len(anchors)
    depth = num_anchors * 5 + num_classes
    target = [[[0.0 for _ in range(depth)] for _ in range(grid)] for _ in range(grid)]
    cx, cy, w, h = xyxy_to_xywh(gt_box)
    cell_size = image_size / grid
    row = min(grid - 1, max(0, int(cy // cell_size)))
    col = min(grid - 1, max(0, int(cx // cell_size)))
    anchor_scores = [wh_iou([w, h], a) for a in anchors]
    anchor_id = max(range(num_anchors), key=lambda i: anchor_scores[i])
    offset = anchor_id * 5
    target[row][col][offset + 0] = cx / cell_size - col
    target[row][col][offset + 1] = cy / cell_size - row
    target[row][col][offset + 2] = w / image_size
    target[row][col][offset + 3] = h / image_size
    target[row][col][offset + 4] = 1.0
    target[row][col][num_anchors * 5 + class_id] = 1.0
    return target, {'row': row, 'col': col, 'anchor_id': anchor_id, 'anchor_scores': anchor_scores}

gt_box = [24.0, 30.0, 58.0, 62.0]
target, info = encode_yolo_target(gt_box, class_id=1)
print(info)
print('encoded vector:', round_list(target[info['row']][info['col']], 3))

エンコードされた座標は、画像全体の絶対座標ではありません。中心はセル内の相対位置、幅と高さは画像サイズで割った値です。実際の YOLO ではアンカー比の対数や sigmoid などを組み合わせますが、担当セル、担当アンカー、objectness、クラスを分ける構造は同じです。

## YOLO の損失は複数の誤差を合わせる

検出では、位置だけを合わせても検出結果としては成立しません。objectness が低ければ検出として出てきません。クラスが外れれば別物として扱われます。損失を座標、objectness、no-object、クラスへ分けると、何が悪いのかを診断できます。

In [ ]:
def zeros_like_target(grid=6, anchors=3, classes=3):
    return [[[0.0 for _ in range(anchors * 5 + classes)] for _ in range(grid)] for _ in range(grid)]


def yolo_loss_components(pred, target, grid=6, anchors=3, classes=3, lambda_coord=5.0, lambda_noobj=0.5):
    coord = 0.0
    obj = 0.0
    noobj = 0.0
    cls = 0.0
    for r in range(grid):
        for c in range(grid):
            cell_has_object = False
            for a in range(anchors):
                off = a * 5
                t_obj = target[r][c][off + 4]
                p_obj = pred[r][c][off + 4]
                if t_obj > 0.5:
                    cell_has_object = True
                    for j in range(4):
                        coord += (pred[r][c][off + j] - target[r][c][off + j]) ** 2
                    obj += (p_obj - t_obj) ** 2
                else:
                    noobj += (p_obj - t_obj) ** 2
            if cell_has_object:
                base = anchors * 5
                for k in range(classes):
                    cls += (pred[r][c][base + k] - target[r][c][base + k]) ** 2
    total = lambda_coord * coord + obj + lambda_noobj * noobj + cls
    return {'coord': coord, 'obj': obj, 'noobj': noobj, 'class': cls, 'total': total}

pred = zeros_like_target()
# 低い背景スコアを全セルに少し入れる
for r in range(6):
    for c in range(6):
        for a in range(3):
            pred[r][c][a * 5 + 4] = 0.04
r, c, a = info['row'], info['col'], info['anchor_id']
off = a * 5
pred[r][c][off:off+5] = [0.42, 0.76, 0.31, 0.30, 0.72]
pred[r][c][15:18] = [0.10, 0.78, 0.12]
print({k: round(v, 4) for k, v in yolo_loss_components(pred, target).items()})

no-object 損失は背景セルと非担当アンカーに効きます。これが弱すぎると、背景にも大量の箱が出ます。強すぎると、物体ありの候補まで消極的になります。検出器の訓練では、この重み付けが精度と再現率のバランスに効きます。

## 予測テンソルを箱へ戻す

モデル出力はそのままでは検出結果ではありません。セルとアンカーの相対値を画像座標の箱へ戻し、objectness とクラス確率を掛けてスコアを作り、しきい値と NMS で整理します。

In [ ]:
def decode_predictions(pred, image_size=IMAGE_SIZE, grid=6, anchors=3, classes=3, score_threshold=0.25):
    detections = []
    cell_size = image_size / grid
    for r in range(grid):
        for c in range(grid):
            class_scores = pred[r][c][anchors * 5:anchors * 5 + classes]
            class_id = max(range(classes), key=lambda k: class_scores[k])
            class_prob = class_scores[class_id]
            for a in range(anchors):
                off = a * 5
                obj = pred[r][c][off + 4]
                score = obj * class_prob
                if score < score_threshold:
                    continue
                cx = (c + pred[r][c][off + 0]) * cell_size
                cy = (r + pred[r][c][off + 1]) * cell_size
                w = pred[r][c][off + 2] * image_size
                h = pred[r][c][off + 3] * image_size
                box = clip_box(xywh_to_xyxy([cx, cy, w, h]), image_size)
                detections.append({'box': box, 'score': score, 'class_id': class_id})
    return detections

# 重複候補を1つ追加して NMS の効果を見る
pred[r][c][0:5] = [0.48, 0.70, 0.35, 0.32, 0.64]
decoded = decode_predictions(pred, score_threshold=0.2)
print('decoded count:', len(decoded))
for det in decoded:
    print(CLASSES[det['class_id']], round(det['score'], 3), round_list(det['box'], 1))

## クラスごとに NMS をかける

異なるクラスの箱は同じ場所にあっても別の候補として扱う場合があります。同じクラス内の重複だけを消すことで、同じ物体を何度も数える問題を減らします。

In [ ]:
def classwise_nms(detections, iou_threshold=0.45):
    result = []
    for class_id in sorted(set(d['class_id'] for d in detections)):
        same_class = [d for d in detections if d['class_id'] == class_id]
        result.extend(nms_single_class(same_class, iou_threshold=iou_threshold))
    return sorted(result, key=lambda d: d['score'], reverse=True)

final_dets = classwise_nms(decoded, iou_threshold=0.45)
print('after NMS:', len(final_dets))
for det in final_dets:
    print(CLASSES[det['class_id']], round(det['score'], 3), 'IoU with gt=', round(iou_xyxy(det['box'], gt_box), 3))

## AP は高スコア順の正解率を見る

Average Precision は、スコアが高い予測から順に見たときの precision と recall の関係を要約します。スコアの高い誤検出が前に来ると AP は下がります。検出では、分類精度だけでなく順位付けの品質も重要です。

In [ ]:
def average_precision(precision, recall):
    ap = 0.0
    for threshold in [i / 10 for i in range(11)]:
        candidates = [p for p, r in zip(precision, recall) if r >= threshold]
        ap += (max(candidates) if candidates else 0.0) / 11.0
    return ap


def evaluate_ap_one_class(predictions, ground_truths, iou_threshold=0.5):
    ordered = sorted(predictions, key=lambda d: d['score'], reverse=True)
    matched = [False for _ in ground_truths]
    tp = []
    fp = []
    for det in ordered:
        best_iou = 0.0
        best_idx = -1
        for i, gt in enumerate(ground_truths):
            value = iou_xyxy(det['box'], gt)
            if value > best_iou:
                best_iou = value
                best_idx = i
        if best_iou >= iou_threshold and best_idx >= 0 and not matched[best_idx]:
            matched[best_idx] = True
            tp.append(1)
            fp.append(0)
        else:
            tp.append(0)
            fp.append(1)
    precision = []
    recall = []
    cum_tp = 0
    cum_fp = 0
    for t, f in zip(tp, fp):
        cum_tp += t
        cum_fp += f
        precision.append(cum_tp / max(cum_tp + cum_fp, 1))
        recall.append(cum_tp / max(len(ground_truths), 1))
    return precision, recall, average_precision(precision, recall)

pred_eval = [
    {'box': [23, 31, 58, 62], 'score': 0.90},
    {'box': [60, 12, 90, 40], 'score': 0.78},
    {'box': [24, 30, 57, 61], 'score': 0.55},
    {'box': [5, 5, 20, 20], 'score': 0.42},
]
gt_eval = [[24, 30, 58, 62], [62, 14, 89, 42]]
precision, recall, ap = evaluate_ap_one_class(pred_eval, gt_eval, iou_threshold=0.5)
print('precision:', round_list(precision, 3))
print('recall:', round_list(recall, 3))
print('AP:', round(ap, 3))

mAP は AP をクラス平均した値です。小さな物体、重なった物体、稀なクラスでは AP が下がりやすくなります。評価表を読むときは、mAP だけで終わらせず、IoU しきい値別、クラス別、サイズ別の失敗を分けて見ます。

## 検出器を読む順序

分類と検出の違いは、出力の構造に現れます。箱の座標表現をそろえ、IoU で位置の当たりを測り、NMS で重複を整理し、YOLO のセル・アンカー・objectness・クラスを読む。最後に AP と mAP でスコア順位まで評価します。この順序で見ると、YOLO の複雑さは個別の部品に分解できます。